In [59]:
from PIL import Image
import mininumpy as mnp


def read_image(path):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    data = list(img.getdata())  # [(R,G,B), ...]

    # Separate channels
    R = [p[0] for p in data]
    G = [p[1] for p in data]
    B = [p[2] for p in data]

    # Concatenate in channel-major order
    flat = R + G + B

    # Now shape is (3, h, w)
    return mnp.Array(flat, shape=(3, h, w), element_type = float)


def save_image(array, path):
    if not isinstance(array, mnp.Array):
        raise TypeError("Expected an Array or np.array instance.")
    if array.shape[0] != 3:
        raise ValueError(f"Expected shape (3, h, w), got {array.shape}")

    c, h, w = array.shape
    data = array._data

    # Split channels
    size_per_channel = h * w
    R = data[0 * size_per_channel: 1 * size_per_channel]
    G = data[1 * size_per_channel: 2 * size_per_channel]
    B = data[2 * size_per_channel: 3 * size_per_channel]

    # Interleave pixel-wise (R0,G0,B0, R1,G1,B1, ...)
    pixels = [(R[i], G[i], B[i]) for i in range(size_per_channel)]

    # Create and save image
    img = Image.new("RGB", (w, h))
    img.putdata(pixels)
    img.save(path)



In [60]:

image = read_image('./sample.png')



In [73]:
gr = mnp.Array([[0.299, 0.587, 0.114] for _ in range(image.size)], shape = image.shape[1:] + (3, 3))

re = mnp.Array([[[1, 0, 0],
                [0, 0.1, 0],
                [0, 0, 0.1]] for _ in range(image.size // 3)], shape = image.shape[1:] + (3, 3))

const = mnp.Array([[[0.1, 0, 0],
                [0, 0.1, 0],
                [0, 0, 0.1]] for _ in range(image.size // 3)], shape = image.shape[1:] + (3, 3))


# save_image(red, './red.jpg')
# save_image(consttrast, './const.jpg')

In [ ]:
A = image.transpose((1, 2, 0))

In [64]:
B = A @ gr.transpose((0, 1, 3, 2))
B = mnp.Array(B._data, shape=B.shape, element_type = int)

In [65]:
save_image(B.transpose((2, 0, 1)), './blabla.png')
        


In [69]:
B = A @ re.transpose((0, 1, 3, 2))
B = mnp.Array(B._data, shape=B.shape, element_type = int)
save_image(B.transpose((2, 0, 1)), './red.png')
        


In [ ]:
B = A @ const.transpose((0, 1, 3, 2))
B = mnp.Array(B._data, shape=B.shape, element_type = int)
save_image(B.transpose((2, 0, 1)), './const.png')
        